# Structured IMDb Join

Create `imdb_structured_joined.csv` from structured IMDb TSV files only. The output columns match `imdb_joined.csv` except that `review` is omitted.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('.')
TITLE_BASICS = DATA_DIR / 'title.basics.tsv'
TITLE_CREW = DATA_DIR / 'title.crew.tsv'
NAME_BASICS = DATA_DIR / 'name.basics.tsv'
OUTPUT_CSV = DATA_DIR / 'imdb_structured_joined.csv'

OUTPUT_COLUMNS = ['movie_id', 'title', 'director', 'year', 'runtime', 'genres']
NA_VALUES = ['\\N']

In [2]:
basics = pd.read_csv(
    TITLE_BASICS,
    sep='\t',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres'],
    dtype=str,
    na_values=NA_VALUES,
    keep_default_na=False,
)

basics = basics.loc[basics['titleType'].eq('movie')].copy()
basics = basics.rename(
    columns={
        'tconst': 'movie_id',
        'primaryTitle': 'title',
        'startYear': 'year',
        'runtimeMinutes': 'runtime',
    }
)

basics = basics[['movie_id', 'title', 'year', 'runtime', 'genres']]
basics = basics.dropna(subset=['movie_id', 'title', 'year', 'runtime', 'genres'])
basics[['year', 'runtime']] = basics[['year', 'runtime']].astype(int)

basics.head()

,movie_id,title,year,runtime,genres
8,tt0000009,Miss Jerry,1894,45,Romance
144,tt0000147,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport"
570,tt0000574,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography"
587,tt0000591,The Prodigal Son,1907,90,Drama
672,tt0000679,The Fairylogue and Radio-Plays,1908,120,"Adventure,Fantasy"


In [3]:
crew = pd.read_csv(
    TITLE_CREW,
    sep='\t',
    usecols=['tconst', 'directors'],
    dtype=str,
    na_values=NA_VALUES,
    keep_default_na=False,
)

crew = crew.rename(columns={'tconst': 'movie_id'})
crew = crew.dropna(subset=['movie_id', 'directors'])
crew = crew.loc[crew['directors'].ne('')].copy()

director_ids = set(crew['directors'].str.split(',').explode().dropna().unique())
len(director_ids)

894917

In [4]:
name_map = {}

for names_chunk in pd.read_csv(
    NAME_BASICS,
    sep='\t',
    usecols=['nconst', 'primaryName'],
    dtype=str,
    na_values=NA_VALUES,
    keep_default_na=False,
    chunksize=500_000,
):
    matched = names_chunk.loc[names_chunk['nconst'].isin(director_ids)]
    name_map.update(zip(matched['nconst'], matched['primaryName']))

len(name_map)

894774

In [5]:
def director_names(director_ids_text: str) -> str:
    names = [name_map.get(director_id) for director_id in director_ids_text.split(',')]
    return ', '.join(name for name in names if name)


crew['director'] = crew['directors'].map(director_names)
crew = crew.loc[crew['director'].ne(''), ['movie_id', 'director']]

joined = basics.merge(crew, on='movie_id', how='inner')
joined = joined[OUTPUT_COLUMNS].sort_values('movie_id').reset_index(drop=True)

joined.head()

,movie_id,title,director,year,runtime,genres
0,tt0000009,Miss Jerry,Alexander Black,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,Enoch J. Rector,1897,100,"Documentary,News,Sport"
2,tt0000574,The Story of the Kelly Gang,Charles Tait,1906,70,"Action,Adventure,Biography"
3,tt0000591,The Prodigal Son,Michel Carré,1907,90,Drama
4,tt0000679,The Fairylogue and Radio-Plays,"Francis Boggs, Otis Turner",1908,120,"Adventure,Fantasy"


In [6]:
joined.to_csv(OUTPUT_CSV, index=False)

print(f'Wrote {len(joined):,} rows to {OUTPUT_CSV}')
joined.head()

Wrote 429,153 rows to imdb_structured_joined.csv


,movie_id,title,director,year,runtime,genres
0,tt0000009,Miss Jerry,Alexander Black,1894,45,Romance
1,tt0000147,The Corbett-Fitzsimmons Fight,Enoch J. Rector,1897,100,"Documentary,News,Sport"
2,tt0000574,The Story of the Kelly Gang,Charles Tait,1906,70,"Action,Adventure,Biography"
3,tt0000591,The Prodigal Son,Michel Carré,1907,90,Drama
4,tt0000679,The Fairylogue and Radio-Plays,"Francis Boggs, Otis Turner",1908,120,"Adventure,Fantasy"
